In [29]:
# Import core libraries
import pandas as pd
import numpy as np
import re
import os

In [30]:
# Import PyTorch
import torch
import torch.nn as nn

In [31]:
# Import sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [32]:
# Import Hugging Face tools
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [33]:
# Optional display settings for easier dataframe viewing
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

### Load Dataset

In [34]:
# Load the 500k sampled dataset from csv
df = pd.read_csv(r"../data/processed/cfpb_sample_500k.csv")

In [35]:
df.shape

(500000, 4)

In [36]:
# Check column names
df.columns.tolist()

['text', 'target', 'text_clean', 'target_grouped']

In [37]:
# Check required columns
df[["text_clean", "target_grouped"]].head()

,text_clean,target_grouped
0,hi i am submitting this xxxx xxxx this isn t any influence and this is not a third party xxxx has low and unfair credit number for me in their report i have 87complained times the problem has not ...,Credit reporting
1,a federal education loan was fraudulently opened in my name i filed an ftc identity theft report a doe loan forgery discharge application and submitted supporting documents despite this the depart...,Debt collection
2,i am writing to have the following information removed from my credit file the items that i need deleted are going to be attached in a word document i am a victim of identity theft i have multiple...,Credit reporting
3,i want this late payment remove from my account and i never been late with this account i working so hard to pay this monthly and this is not acceptable see the documents attached,Credit card
4,i am disputing several items on my experian credit report that appear to be inaccurate or incomplete the details of the accounts in question are as follows 1 charged off accounts xxxx xxxx xxxx xx...,Credit reporting


In [38]:
# Check missing values in required columns
df[["text_clean", "target_grouped"]].isnull().sum()

text_clean        0
target_grouped    0
dtype: int64

### Create label mapping

In [39]:
# Create label mappings
label2id = {label: i for i, label in enumerate(sorted(df["target_grouped"].unique()))}
id2label = {i: label for label, i in label2id.items()}

In [40]:
# Create numeric label column
df["label"] = df["target_grouped"].map(label2id)

In [41]:
label2id

{'Auto loan': 0,
 'Bank account': 1,
 'Credit card': 2,
 'Credit reporting': 3,
 'Debt collection': 4,
 'Money transfer': 5,
 'Mortgage': 6,
 'Other': 7,
 'Personal loan': 8,
 'Student loan': 9}

In [42]:
id2label

{0: 'Auto loan',
 1: 'Bank account',
 2: 'Credit card',
 3: 'Credit reporting',
 4: 'Debt collection',
 5: 'Money transfer',
 6: 'Mortgage',
 7: 'Other',
 8: 'Personal loan',
 9: 'Student loan'}

In [43]:
# Preview labels
df[["target_grouped", "label"]].drop_duplicates().sort_values("label")

,target_grouped,label
199,Auto loan,0
10,Bank account,1
3,Credit card,2
0,Credit reporting,3
1,Debt collection,4
16,Money transfer,5
42,Mortgage,6
435,Other,7
15,Personal loan,8
161,Student loan,9


### Sample data for stronger transformation training

In [44]:
df_transformer = df[["text_clean", "target_grouped", "label"]].sample(
    n=250000,
    random_state = 42
)

In [45]:
df_transformer.isnull().sum()

text_clean        0
target_grouped    0
label             0
dtype: int64

In [46]:
df_transformer.shape

(250000, 3)

In [47]:
df_transformer["target_grouped"].value_counts()

target_grouped
Credit reporting    167210
Debt collection      27473
Credit card          15355
Bank account         12472
Mortgage              9394
Money transfer        7661
Student loan          3991
Auto loan             3223
Personal loan         2893
Other                  328
Name: count, dtype: int64

### Create Train-Test-Split

In [48]:
# split data in test and train

train_df, test_df = train_test_split(
    df_transformer,
    test_size = 0.2,
    random_state=42,
    stratify=df_transformer["label"]
)

In [49]:
# check shape

print(train_df.shape, test_df.shape)

(200000, 3) (50000, 3)


In [50]:
# check label distribution in train split

train_df["target_grouped"].value_counts()

target_grouped
Credit reporting    133768
Debt collection      21979
Credit card          12284
Bank account          9978
Mortgage              7515
Money transfer        6129
Student loan          3193
Auto loan             2578
Personal loan         2314
Other                  262
Name: count, dtype: int64

In [51]:
# check label distribution in test split

test_df["target_grouped"].value_counts()

target_grouped
Credit reporting    33442
Debt collection      5494
Credit card          3071
Bank account         2494
Mortgage             1879
Money transfer       1532
Student loan          798
Auto loan             645
Personal loan         579
Other                  66
Name: count, dtype: int64

### Convert to transformers dataset

In [52]:
# convert train and test dataframes to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df[["text_clean", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text_clean", "label"]])

In [53]:
# check names

print(train_dataset.column_names)

['text_clean', 'label', '__index_level_0__']


In [54]:
print(test_dataset.column_names)

['text_clean', 'label', '__index_level_0__']


### set the model name

[Model Link](https://huggingface.co/FacebookAI/roberta-base)

In [55]:
# set the stronger transformer backbone
model_name = "roberta-base"

In [56]:
# Load Tokenizer for RoBERTa
tokenizer = AutoTokenizer.from_pretrained(model_name)

### Tokenize

In [57]:
# create the tokenization function

def tokenize_function(examples):
    return tokenizer(
        examples["text_clean"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

#### Apply tokenizer

In [58]:
# apply tokenization to train and test dataset

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batch_size=True)

Map:   0%|          | 0/200000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [59]:
train_dataset.column_names

['text_clean', 'label', '__index_level_0__', 'input_ids', 'attention_mask']

In [60]:
# Remove extra columns if present
cols_to_remove = [col for col in ["text_clean", "__index_level_0__"] if col in train_dataset.column_names]
train_dataset = train_dataset.remove_columns(cols_to_remove)
test_dataset = test_dataset.remove_columns(cols_to_remove)

In [61]:
# set pytorch format for pytorch
train_dataset.set_format("torch")
test_dataset.set_format("torch")

In [62]:
train_dataset

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 200000
})

In [63]:
# check final dataset columns and one sample

print(train_dataset.column_names)
print(train_dataset[0])

['label', 'input_ids', 'attention_mask']
{'label': tensor(9), 'input_ids': tensor([    0,   118,   475,   145,  1340,    13,    10,  2541,    77,   939,
          174,     5,  2737,  3023, 46628,  3023, 46628,  1564,    14,   939,
           74,    45,    28,   441,     7,  2725,   528,     7,   474,  1486,
            8,   939,  1882,    66,     9,     5,   586,   648,    51,   439,
          789,     8, 12069,     5,  2541,    25,   114,   939,   393,  2294,
         5190,   939,    21,  4768,    25,  1085,   115,    28,   626,    53,
           89,  1302,     7,    28,   402,     2,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,    

### Compute class weights from the training splits

In [64]:
# get sorted label ids from the training data

class_ids = sorted(train_df["label"].unique())

In [65]:
class_ids

[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9)]

In [66]:
# Compute balanced class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(class_ids),
    y=train_df["label"].values
)

In [67]:
class_weights

array([ 7.7579519 ,  2.0044097 ,  1.62813416,  0.14951259,  0.90995951,
        3.26317507,  2.66134398, 76.33587786,  8.64304235,  6.26370185])

In [68]:
# Convert class weights to PyTorch tensor
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

In [69]:
class_weights_tensor

tensor([ 7.7580,  2.0044,  1.6281,  0.1495,  0.9100,  3.2632,  2.6613, 76.3359,
         8.6430,  6.2637])

#### Load RoBERTa model

In [70]:
# Load RoBERTa model for classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#### Create a weighted Trainer class

In [ ]:
# Create a custom Trainer that uses weighted cross-entropy loss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor.to(model.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

#### Create the metric function

In [72]:
# Create metric function for evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, preds)
    weighted_f1 = f1_score(labels, preds, average="weighted", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)

    return {
        "accuracy": accuracy,
        "weighted_f1": weighted_f1,
        "macro_f1": macro_f1
    }

### Training Arguments and Trainer setup

In [73]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./roberta_results_250k",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="weighted_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none"
)

If memory is tight just use:

- per_device_train_batch_size=2
- per_device_eval_batch_size=2
- gradient_accumulation_steps=4

In [74]:
# Create weighted trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

#### device check

### Sanity check

In [76]:
# Check one batch sample
print(train_dataset[0])

{'label': tensor(9), 'input_ids': tensor([    0,   118,   475,   145,  1340,    13,    10,  2541,    77,   939,
          174,     5,  2737,  3023, 46628,  3023, 46628,  1564,    14,   939,
           74,    45,    28,   441,     7,  2725,   528,     7,   474,  1486,
            8,   939,  1882,    66,     9,     5,   586,   648,    51,   439,
          789,     8, 12069,     5,  2541,    25,   114,   939,   393,  2294,
         5190,   939,    21,  4768,    25,  1085,   115,    28,   626,    53,
           89,  1302,     7,    28,   402,     2,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,  

In [77]:
# Check model device
print(next(model.parameters()).device)

cpu


In [78]:
# Check GPU availability
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

False
CPU only


### Train Model

In [ ]:
# Train the RoBERTa model

trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy,Weighted F1,Macro F1
1,1.533298,0.766909,0.004000,0.889540,0.889103,0.687052


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


### Predictions

In [ ]:
# Generate predictions on the test dataset
predictions = trainer.predict(test_dataset)

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
# Convert logits to predicted class ids
y_pred_roberta = np.argmax(predictions.predictions, axis=1)

In [ ]:
# Extract true labels
y_true_roberta = predictions.label_ids

In [ ]:
# Convert numeric ids back to label names
y_pred_roberta_labels = [id2label[i] for i in y_pred_roberta]
y_true_roberta_labels = [id2label[i] for i in y_true_roberta]

In [ ]:
# Print classification report
print(classification_report(y_true_roberta_labels, y_pred_roberta_labels, zero_division=0))

In [ ]:
# Calculate summary metrics
roberta_accuracy = accuracy_score(y_true_roberta_labels, y_pred_roberta_labels)
roberta_weighted_f1 = f1_score(y_true_roberta_labels, y_pred_roberta_labels, average="weighted", zero_division=0)
roberta_macro_f1 = f1_score(y_true_roberta_labels, y_pred_roberta_labels, average="macro", zero_division=0)

print("RoBERTa Accuracy:", roberta_accuracy)
print("RoBERTa Weighted F1:", roberta_weighted_f1)
print("RoBERTa Macro F1:", roberta_macro_f1)

# Save the fine-tuned RoBERTa model and tokenizer
trainer.save_model("models/roberta_complaint_classifier")
tokenizer.save_pretrained("models/roberta_complaint_classifier")